# Option 2 — Symbolic Conditioned MIDI Generation

**Task**: Given a 4-second MIDI prefix, autoregressively generate a 4-second continuation using a small GPT-2 Transformer trained on REMI tokens from MAESTRO v3.0.0.

**Pipeline**: Setup → Download MAESTRO → Build REMI token cache → Build windows → Train → Generate → Evaluate

## 1. Environment Setup

In [1]:
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Running on Colab:', IN_COLAB)

if IN_COLAB:
    REPO_URL = 'https://github.com/archerop/CSE_253.git'
    REPO_DIR = Path('/content/CSE_253')
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        result = !git -C {REPO_DIR} pull --ff-only
        print('\n'.join(result))
        # Flush stale cached modules so re-imports read the updated files
        stale = [k for k in sys.modules if k == 'app' or k.startswith('app.')]
        for k in stale:
            del sys.modules[k]
        importlib.invalidate_caches()
        if stale:
            print(f'Flushed {len(stale)} cached module(s)')
    ASSIGNMENT_ROOT = REPO_DIR / 'assignment2'
else:
    ASSIGNMENT_ROOT = Path('__file__').resolve().parent if '__file__' in dir() else Path('.').resolve()
    for p in [ASSIGNMENT_ROOT] + list(ASSIGNMENT_ROOT.parents):
        if (p / 'app').exists():
            ASSIGNMENT_ROOT = p
            break

print('Assignment root:', ASSIGNMENT_ROOT)
os.chdir(ASSIGNMENT_ROOT)
if str(ASSIGNMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_ROOT))

remote: Enumerating objects: 18, done.
remote: Counting objects:   5% (1/18)
remote: Counting objects:  11% (2/18)
remote: Counting objects:  16% (3/18)
remote: Counting objects:  22% (4/18)
remote: Counting objects:  27% (5/18)
remote: Counting objects:  33% (6/18)
remote: Counting objects:  38% (7/18)
remote: Counting objects:  44% (8/18)
remote: Counting objects:  50% (9/18)
remote: Counting objects:  55% (10/18)
remote: Counting objects:  61% (11/18)
remote: Counting objects:  66% (12/18)
remote: Counting objects:  72% (13/18)
remote: Counting objects:  77% (14/18)
remote: Counting objects:  83% (15/18)
remote: Counting objects:  88% (16/18)
remote: Counting objects:  94% (17/18)
remote: Counting objects: 100% (18/18)
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects:  25% (1/4)
remote: Compressing objects:  50% (2/4)
remote: Compressing objects:  75% (3/4)
remote: Compressing objects: 100% (4/4)
remote: Compressing objects: 100% (4/4), done.
remote: Total 1

## 2. Install Dependencies

In [ ]:
import importlib, subprocess

def need(pkg):
    return importlib.util.find_spec(pkg) is None

pkgs = [p for p in ['pretty_midi', 'torch', 'pandas', 'tqdm', 'scipy', 'miditok', 'symusic', 'transformers'] if need(p)]
if pkgs:
    print('Installing:', pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
else:
    print('All dependencies already installed.')

## 3. Download MAESTRO MIDI Dataset

In [3]:
import urllib.request, zipfile

MAESTRO_ROOT = ASSIGNMENT_ROOT / 'data' / 'maestro-v3.0.0'

if (MAESTRO_ROOT / 'maestro-v3.0.0.csv').exists():
    print('MAESTRO already present at', MAESTRO_ROOT)
else:
    URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
    ZIP = ASSIGNMENT_ROOT / 'data' / 'downloads' / 'maestro-v3.0.0-midi.zip'
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists():
        print('Downloading MAESTRO MIDI (~57 MB)...')
        urllib.request.urlretrieve(URL, ZIP)
    print('Extracting...')
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ASSIGNMENT_ROOT / 'data')
    print('Done.')

print(f'MIDI files: {len(list(MAESTRO_ROOT.rglob("*.midi")))}')

MAESTRO already present at /content/CSE_253/assignment2/data/maestro-v3.0.0
MIDI files: 1276


## 4. Imports

In [ ]:
import json, pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
import pretty_midi
from tqdm.auto import tqdm

from app.shared.config import (
    MAESTRO_ROOT, OPTION2_CACHE_DIR, OPTION2_OUTPUT_DIR, CHECKPOINT_DIR,
    OPTION2_FRAME_RATE, OPTION2_PREFIX_SECONDS, OPTION2_CONTINUATION_SECONDS,
    OPTION2_BATCH_SIZE, OPTION2_LEARNING_RATE, OPTION2_WEIGHT_DECAY,
    OPTION2_MAX_EPOCHS, OPTION2_PATIENCE, OPTION2_PREFIX_MAX_LEN, OPTION2_CONT_MAX_LEN,
)
from app.shared.metadata import load_maestro_metadata, validate_maestro_paths
from app.option2.symbolic_dataset import (
    build_tokenizer, precache_tokens, build_token_window_index, SymbolicDataset, get_datasets,
)
from app.option2.symbolic_models import build_gpt2_model, generate_tokens, CopyLastPatternBaseline
from app.option2.symbolic_train import train, load_best_checkpoint
from app.option2.symbolic_generate import (
    extract_prefix, generate_conditioned, save_symbolic_conditioned, tokens_to_pianoroll,
)
from app.option2.symbolic_eval import evaluate_generation, evaluate_token_generation, print_metrics

print('Imports OK')

In [5]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)

Device: cuda


## 5. Google Drive Setup (Colab only)
Mounts Google Drive and routes the checkpoint there so it survives session resets.

In [6]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_DIR = Path('/content/drive/MyDrive/CSE253')
    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PATH = GDRIVE_DIR / 'option2_best.pt'
    print('Checkpoint → Google Drive:', CKPT_PATH)
else:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PATH = CHECKPOINT_DIR / 'option2_best.pt'
    print('Checkpoint → local:', CKPT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint → Google Drive: /content/drive/MyDrive/CSE253/option2_best.pt


## 6. Build REMI Tokenizer & Pre-cache Tokens
Tokenizes all 1,276 MIDI files once using the REMI tokenizer and saves per-file `.pkl` caches. Safe to re-run — skips existing files.

In [ ]:
tokenizer = build_tokenizer()
print(f'Tokenizer ready. vocab_size={tokenizer.vocab_size}  PAD={tokenizer["PAD_None"]}  BOS={tokenizer["BOS_None"]}  EOS={tokenizer["EOS_None"]}')
precache_tokens(tokenizer)

## 7. Build Token Window Index
Slides a window of `PREFIX_MAX_LEN + CONT_MAX_LEN = 512` tokens over each MIDI's token sequence with stride 128.

In [ ]:
OPTION2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_windows = build_token_window_index('train',      tokenizer, cache_path=OPTION2_CACHE_DIR / 'train_token_windows.pkl')
val_windows   = build_token_window_index('validation', tokenizer, cache_path=OPTION2_CACHE_DIR / 'val_token_windows.pkl')
test_windows  = build_token_window_index('test',       tokenizer, cache_path=OPTION2_CACHE_DIR / 'test_token_windows.pkl')

print(f'Windows — train: {len(train_windows):,}  val: {len(val_windows):,}  test: {len(test_windows):,}')

## 8. Dataset & DataLoader

In [ ]:
train_ds = SymbolicDataset(train_windows, tokenizer)
val_ds   = SymbolicDataset(val_windows,   tokenizer)

prefix, cont = train_ds[0]
print(f'prefix shape: {tuple(prefix.shape)}  dtype: {prefix.dtype}')
print(f'cont shape:   {tuple(cont.shape)}   dtype: {cont.dtype}')
print(f'non-pad prefix tokens: {(prefix != tokenizer["PAD_None"]).sum().item()}')

In [10]:
PIN_MEMORY  = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if IN_COLAB else 4

train_loader = DataLoader(train_ds, batch_size=OPTION2_BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=OPTION2_BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train batches: {len(train_loader):,}  Val batches: {len(val_loader):,}')

Train batches: 8,853  Val batches: 1,078


## 9. Model

| Component | Value |
|-----------|-------|
| Architecture | GPT-2 (causal LM, HuggingFace) |
| Tokenization | REMI (vocab ≈ 347 tokens) |
| n_layer | 6 |
| n_head | 8 |
| n_embd | 384 |
| Parameters | ~11M |
| Loss | CrossEntropyLoss (prefix masked with -100) |
| Training | Teacher forcing over `[prefix \| continuation[:-1]]` |

In [ ]:
model = build_gpt2_model(vocab_size=tokenizer.vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}  ({n_params/1e6:.1f}M)')

## 10. Training

In [ ]:
history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    checkpoint_path=CKPT_PATH,
)

OPTION2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OPTION2_OUTPUT_DIR / 'train_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Training complete. Checkpoint saved to:', CKPT_PATH)

## 11. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='Train')
plt.plot(epochs, history['val_loss'],   label='Val')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Option 2 Training Loss (GPT-2 + REMI)')
plt.legend()
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'loss_curve.png', dpi=150)
plt.show()

## 12. Generate `symbolic_conditioned.mid`

In [ ]:
df = load_maestro_metadata(MAESTRO_ROOT)
df = validate_maestro_paths(df)
test_df = df[(df['split'] == 'test') & df['midi_exists']].sort_values('duration').reset_index(drop=True)
row = test_df.iloc[len(test_df) // 2]
MIDI_PATH = row['midi_path']
print(f"Using: {row['composer']} — {row['title']}  ({row['duration']:.1f}s)")

Using: Ludwig van Beethoven — Sonata No. 16 Op. 31 No. 1 in G Major, I. Allegro vivace  (306.1s)


In [ ]:
model = load_best_checkpoint(build_gpt2_model(vocab_size=tokenizer.vocab_size), CKPT_PATH, DEVICE)

output_path = save_symbolic_conditioned(
    prefix_midi_path=MIDI_PATH,
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
)
print('Output saved to:', output_path)

## 13. Evaluation — Model vs Baseline

In [ ]:
import symusic
from app.option2.symbolic_dataset import _trim_score

# Extract prefix tokens and ground-truth continuation tokens
prefix_ids = extract_prefix(MIDI_PATH, tokenizer)

score    = symusic.Score(MIDI_PATH)
cont_pm  = _trim_score(score, OPTION2_PREFIX_SECONDS, OPTION2_PREFIX_SECONDS + OPTION2_CONTINUATION_SECONDS)
gt_seqs  = tokenizer.encode(cont_pm)
gt_ids   = gt_seqs[0].ids if gt_seqs else []

# Model generation
cont_ids = generate_conditioned(model, prefix_ids, OPTION2_CONT_MAX_LEN, DEVICE)
model_roll   = tokens_to_pianoroll(cont_ids.tolist(), tokenizer)
gt_roll      = tokens_to_pianoroll(gt_ids,            tokenizer)

# Baseline: cycle last prefix tokens
baseline_ids  = CopyLastPatternBaseline().generate(prefix_ids.unsqueeze(0), OPTION2_CONT_MAX_LEN).squeeze(0)
baseline_roll = tokens_to_pianoroll(baseline_ids.tolist(), tokenizer)

print('=== Model ===')
print_metrics(evaluate_generation(model_roll, gt_roll))

print('=== Baseline (copy last pattern) ===')
print_metrics(evaluate_generation(baseline_roll, gt_roll))

In [ ]:
import matplotlib.pyplot as plt

prefix_roll = tokens_to_pianoroll(prefix_ids.tolist(), tokenizer,
                                  duration_seconds=OPTION2_PREFIX_SECONDS)
combined = np.concatenate([prefix_roll, model_roll], axis=0).T

split_frame = len(prefix_roll)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(combined, aspect='auto', origin='lower', cmap='Blues', interpolation='nearest')
ax.axvline(split_frame - 0.5, color='red', linewidth=1.5, label='prefix | generated')
ax.set_xlabel('Frame (40 fps)')
ax.set_ylabel('Pitch (MIDI 21–108)')
ax.set_title('Piano-roll: prefix (left) + generated continuation (right)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(OPTION2_OUTPUT_DIR / 'pianoroll_visualization.png', dpi=150)
plt.show()

## Done

| Output | Location |
|--------|----------|
| Generated MIDI | `outputs/option2/symbolic_conditioned.mid` |
| Best checkpoint | Google Drive `CSE253/option2_best.pt` (Colab) or `outputs/checkpoints/option2_best.pt` (local) |
| Loss curve | `outputs/option2/loss_curve.png` |
| Piano-roll plot | `outputs/option2/pianoroll_visualization.png` |